In [ ]:
import pandas as pd
import pm4py

INPUT_CSV = (
    "../../data/processed/CTB/"
    "s6_no_target_block_discovery.csv"
)

OUTPUT_XES = (
    "../../data/processed/CTB/xes_files/"
    "s5_sample_25.000_no_target_block_discovery.xes"
)

N_CASES = 25000
RANDOM_STATE = 42

df = pd.read_csv(INPUT_CSV)

# ----------------------------------------------------------
# TIMESTAMPS
# ----------------------------------------------------------

for col in [
    "enabled:timestamp",
    "start:timestamp",
    "time:timestamp"
]:
    df[col] = pd.to_datetime(
        df[col],
        errors="coerce"
    )

# ==========================================================
# CASE SAMPLING
# ==========================================================

print("Sampling cases...")

case_ids = (
    df["case:concept:name"]
    .drop_duplicates()
    .sample(
        n=min(
            N_CASES,
            df["case:concept:name"].nunique()
        ),
        random_state=RANDOM_STATE
    )
)

df = df[
    df["case:concept:name"].isin(case_ids)
].copy()

print(
    f"Sampled cases: "
    f"{df['case:concept:name'].nunique():,}"
)

# ----------------------------------------------------------
# PM4PY FORMAT
# ----------------------------------------------------------

df = pm4py.format_dataframe(
    df,
    case_id="case:concept:name",
    activity_key="concept:name",
    timestamp_key="time:timestamp"
)

# ----------------------------------------------------------
# CONVERT TO EVENT LOG
# ----------------------------------------------------------

event_log = pm4py.convert_to_event_log(
    df
)

# ----------------------------------------------------------
# EXPORT
# ----------------------------------------------------------

pm4py.write_xes(
    event_log,
    OUTPUT_XES
)

print(
    f"Saved:\n{OUTPUT_XES}"
)



  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing




Sampling cases...
Sampled cases: 20,000


c:\Users\Grigat-J\AppData\Local\miniconda3\envs\prosit_test\lib\site-packages\pm4py\utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(
c:\Users\Grigat-J\AppData\Local\miniconda3\envs\prosit_test\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
exporting log, completed traces :: 100%|██████████| 20000/20000 [00:14<00:00, 1362.32it/s]

Saved:
../../data/processed/CTB/xes_files/s5_sample_25.000_no_target_block_discovery.xes


In [1]:
import warnings
warnings.filterwarnings("ignore")

import pm4py
import pm4py.objects.log.importer.xes.importer as xes_importer
from prosit import SimulatorParameters, SimulatorEngine
from datetime import datetime

# 1. Load event log
log = xes_importer.apply("../../data/processed/CTB/xes_files/"
    "s5_sample_30.000_eventlog_one_block_target_features.xes")
print("loaded .xes")



  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing


parsing log, completed traces :: 100%|██████████| 30000/30000 [00:16<00:00, 1811.18it/s]

loaded .xes


In [3]:
# =====================================================
# INSPECT AVAILABLE ATTRIBUTES
# =====================================================

first_event = log[0][0]

print("Event attributes:")
for k in first_event.keys():
    print(" ", k)

Event attributes:
  concept:name
  org:resource
  enabled:timestamp
  start:timestamp
  time:timestamp
  process_flow_type
  n_containers
  n_stops
  n_deliveries
  n_receives
  has_hazardous
  has_reefer
  full_ratio
  visit_complexity
  gate_demand
  rmg_demand
  vc_demand
  mt_demand
  gate_utilization
  rmg_utilization
  vc_utilization
  mt_utilization
  target_block
  target_utilization
  target_demand
  target_utilization_bin
  target_demand_bin


In [ ]:
# Keep only basic process features =====================================================
# KEEP ONLY BASIC PROCESS ATTRIBUTES
# =====================================================

from pm4py.objects.log.obj import EventLog, Trace

KEEP = {
    "concept:name",
    "org:resource",
    "time:timestamp",
    "start:timestamp",
    "enabled:timestamp"
}

filtered_log = EventLog()

for trace in log:

    new_trace = Trace()
    new_trace.attributes = trace.attributes.copy()

    for event in trace:

        new_event = {
            k: v
            for k, v in event.items()
            if k in KEEP
        }

        new_trace.append(new_event)

    filtered_log.append(new_trace)

log = filtered_log

print("Remaining attributes:")
print(sorted(KEEP))


In [ ]:
# keep process and target features =====================================================
# KEEP PROCESS + TARGET FEATURES
# =====================================================

from pm4py.objects.log.obj import EventLog, Trace

KEEP = {
    "concept:name",
    "org:resource",
    "time:timestamp",
    "start:timestamp",
    "enabled:timestamp",

    "target_utilization_bin",
    "target_demand_bin"
}

filtered_log = EventLog()

for trace in log:

    new_trace = Trace()
    new_trace.attributes = trace.attributes.copy()

    for event in trace:

        new_event = {
            k: v
            for k, v in event.items()
            if k in KEEP
        }

        new_trace.append(new_event)

    filtered_log.append(new_trace)

log = filtered_log

print("Remaining attributes:")
print(sorted(KEEP))

In [ ]:
# Full log for Discovery (no target block)
# =====================================================

from pm4py.objects.log.obj import EventLog, Trace

KEEP = {
    "concept:name",
    "org:resource",
    "time:timestamp",
    "start:timestamp",
    "enabled:timestamp",

    "process_flow_type",
    "n_containers",
    "n_stops",
    "n_deliveries",
    "n_receives",
    "has_hazardous",
    "has_reefer",
    "full_ratio",
    "visit_complexity",
    "gate_demand",
    "rmg_demand",
    "vc_demand",
    "mt_demand",
    "gate_utilization",
    "rmg_utilization",
    "vc_utilization",
    "mt_utilization",
    #"target_block",
    "target_utilization",
    "target_demand",
    "target_utilization_bin",
    "target_demand_bin",
    "target_rank",
    "target_rank_group"
}

filtered_log = EventLog()

for trace in log:

    new_trace = Trace()
    new_trace = Trace(
    attributes=trace.attributes.copy()
    )

    for event in trace:

        new_event = {
            k: v
            for k, v in event.items()
            if k in KEEP
        }

        new_trace.append(new_event)

    filtered_log.append(new_trace)

log = filtered_log

print("Remaining attributes:")
print(sorted(KEEP))

Remaining attributes:
['concept:name', 'enabled:timestamp', 'full_ratio', 'gate_demand', 'gate_utilization', 'has_hazardous', 'has_reefer', 'mt_demand', 'mt_utilization', 'n_containers', 'n_deliveries', 'n_receives', 'n_stops', 'org:resource', 'process_flow_type', 'rmg_demand', 'rmg_utilization', 'start:timestamp', 'target_demand', 'target_demand_bin', 'target_utilization', 'target_utilization_bin', 'time:timestamp', 'vc_demand', 'vc_utilization', 'visit_complexity']


In [6]:
# Split: use 80% for discovery, compare simulation against remaining 20%
n_cases = len(log)
#train_log = log[:int(n_cases * 0.8)]
#test_log  = log[int(n_cases * 0.8):]
from pm4py.objects.log.obj import EventLog

n_cases = len(log)

train_log = EventLog(
    list(log[:int(n_cases * 0.8)])
)

test_log = EventLog(
    list(log[int(n_cases * 0.8):])
)
print(f"Split into training set ({len(train_log)} cases) and test set ({len(test_log)} cases)")

# Discover from training set
net, im, fm = pm4py.discover_petri_net_inductive(train_log)
params = SimulatorParameters(net, im, fm)


Split into training set (24000 cases) and test set (6000 cases)


In [7]:
df_train = pm4py.convert_to_dataframe(
    train_log
)

print(df_train.columns.tolist())

['concept:name', 'org:resource', 'enabled:timestamp', 'start:timestamp', 'time:timestamp', 'process_flow_type', 'n_containers', 'n_stops', 'n_deliveries', 'n_receives', 'has_hazardous', 'has_reefer', 'full_ratio', 'visit_complexity', 'gate_demand', 'rmg_demand', 'vc_demand', 'mt_demand', 'gate_utilization', 'rmg_utilization', 'vc_utilization', 'mt_utilization', 'target_utilization', 'target_demand', 'target_utilization_bin', 'target_demand_bin', 'case:concept:name']


In [8]:
import pm4py

df_train = pm4py.convert_to_dataframe(
    train_log
)

from collections import Counter

c = Counter(df_train.columns)

duplicates = {
    k:v
    for k,v in c.items()
    if v > 1
}

print(duplicates)
cols = list(df_train.columns)

print(
    len(cols),
    len(set(cols))
)

{}
27 27


In [9]:
# Uses use_workload_features=True
params.discover_from_eventlog(
    train_log,
    max_depth_tree=10,
    min_samples_leaf_cv=[5,10,20],
    random_state=42,
    verbose=True,
    use_workload_features=True
)
print(f"Discovered parameters: {params}")
# Simulate the same number of cases as the test set
engine = SimulatorEngine(params)


Resources discovery...
Data attributes discovery...
PATCHED 1.0 script is running
Feature discovery...


100%|██████████| 24000/24000 [00:19<00:00, 1209.27it/s]


Transition Probabilities discovery...


transition models: 100%|██████████| 36/36 [00:51<00:00,  1.43s/it]


Resource Weights discovery...


...

Calendars discovery...
Execution Time discovery...


exec-time models: 100%|██████████| 11/11 [00:10<00:00,  1.05it/s]


Waiting Time discovery...


waiting-time models: 100%|██████████| 26/26 [00:18<00:00,  1.43it/s]


Arrival Time discovery...
Discovered parameters: <prosit.simulator.SimulatorParameters object at 0x000002245CB1D480>


In [10]:
sim_log = engine.apply(
    n_traces=len(train_log),
    #t_start=datetime(2024, 1, 1, 8, 0, 0)
)
sim_log.to_csv("../../data/processed/CTB/prosit_simulations/sim_log_s5_sample_30.000_depth10.csv")
print(f"Simulated {len(sim_log)} events across {sim_log['case:concept:name'].nunique()} cases")


Simulating Cases: 100%|██████████| 24000/24000 [00:19<00:00, 1230.24it/s]


Simulated 71631 events across 24000 cases


In [11]:
params.to_json("../../data/processed/CTB/prosit_params/sim_log_s5_sample_30.000_depth10_params.json")

In [12]:
import pickle

with open(
    "../../data/processed/CTB/prosit_params/sim_log_s5_sample_30.000_depth10_params.pkl",
    "wb"
) as f:

    pickle.dump(
        params,
        f
    )

JSON INSPIZIEREN

In [ ]:
import json

with open(
    "../../data/processed/CTB/prosit_params/sim_log_s5_sample_30.000_depth10_params.json",
    "r",
    encoding="utf-8"
) as f:

    model = json.load(f)

print(type(model))

if isinstance(model, dict):

    print(
        model.keys()
    )

<class 'dict'>
dict_keys(['rules_mode', 'use_workload_features', 'transition_params', 'resource_params', 'arrival_params', 'execution_time_params', 'waiting_time_params', 'data_attribute_params'])


In [17]:
from pprint import pprint

for k, v in model["resource_params"].items():

    print("\n")
    print("=" * 80)

    print(k)

    print("=" * 80)

    if isinstance(v, dict):

        print(
            "DICT KEYS:"
        )

        print(
            list(v.keys())[:20]
        )

    elif isinstance(v, list):

        print(
            f"LIST LENGTH: {len(v)}"
        )

        if len(v):

            print(
                "FIRST ELEMENT:"
            )

            pprint(v[0])

    else:

        pprint(v)



resources
LIST LENGTH: 26
FIRST ELEMENT:
'Res.GateIn'


max_concurrency
DICT KEYS:
['Res.GateIn', 'Res.GateOut', 'LL', 'T13', 'T14', 'T24', 'T16', 'T17', 'T15', 'T12', 'T21', 'T20', 'T18', 'T11', 'T23', 'T27', 'T22', 'T10', 'T06', 'T09']


act_to_resources
DICT KEYS:
['HO2_receive', 'RMG_receive', 'HO2_delivery', 'Gate Out', 'LL_receive', 'RMG_mixed', 'RMG_delivery', 'HO2_mixed', 'Gate In', 'LL_mixed', 'LL_delivery']


resource_weights
DICT KEYS:
['Res.GateIn', 'Res.GateOut', 'LL', 'T13', 'T14', 'T24', 'T16', 'T17', 'T15', 'T12', 'T21', 'T20', 'T18', 'T11', 'T23', 'T27', 'T22', 'T10', 'T06', 'T09']


calendars
DICT KEYS:
['Res.GateIn', 'Res.GateOut', 'LL', 'T13', 'T14', 'T24', 'T16', 'T17', 'T15', 'T12', 'T21', 'T20', 'T18', 'T11', 'T23', 'T27', 'T22', 'T10', 'T06', 'T09']


In [18]:
from pprint import pprint

for k, v in model.items():

    print("\n" + "=" * 80)

    print(k)

    print("=" * 80)

    print(type(v))

    if isinstance(v, dict):

        print(v.keys())


rules_mode
<class 'bool'>

use_workload_features
<class 'bool'>

transition_params
<class 'dict'>
dict_keys(['transition_weights'])

resource_params
<class 'dict'>
dict_keys(['resources', 'max_concurrency', 'act_to_resources', 'resource_weights', 'calendars'])

arrival_params
<class 'dict'>
dict_keys(['arrival_calendar', 'arrival_time_distributions'])

execution_time_params
<class 'dict'>
dict_keys(['execution_time_distributions'])

waiting_time_params
<class 'dict'>
dict_keys(['waiting_time_distributions'])

data_attribute_params
<class 'dict'>
dict_keys(['label_data_attributes', 'label_data_attributes_categorical', 'attribute_values_label_categorical', 'distribution_data_attributes'])


In [19]:
from pprint import pprint

pprint(
    model["resource_params"]["resource_weights"]
)

{'HO2': 1.0,
 'LL': 1.0,
 'Res.GateIn': 1.0,
 'Res.GateOut': 1.0,
 'T06': {'0': {'children': {'false': 2, 'true': 1},
               'feature': 'target_utilization',
               'threshold': 0.51},
         '1': {'value': 0.186},
         '2': {'value': 0.01}},
 'T07': {'0': {'children': {'false': 2, 'true': 1},
               'feature': 'target_utilization',
               'threshold': 0.57},
         '1': {'value': 0.122},
         '2': {'value': 0.002}},
 'T08': {'0': {'children': {'false': 2, 'true': 1},
               'feature': 'target_utilization',
               'threshold': 0.485},
         '1': {'value': 0.211},
         '2': {'value': 0.005}},
 'T09': {'0': {'children': {'false': 2, 'true': 1},
               'feature': 'target_utilization',
               'threshold': 0.521},
         '1': {'value': 0.121},
         '2': {'value': 0.021}},
 'T10': {'0': {'children': {'false': 2, 'true': 1},
               'feature': 'target_utilization',
               'threshold': 0.518

In [21]:
#TREE DUMP
import json
from pprint import pprint

def recursive_print(
    obj,
    depth=0,
    max_depth=4
):

    if depth > max_depth:
        return

    indent = "  " * depth

    if isinstance(obj, dict):

        for k, v in obj.items():

            print(
                f"{indent}{k}"
            )

            recursive_print(
                v,
                depth + 1,
                max_depth
            )

    elif isinstance(obj, list):

        print(
            f"{indent}LIST[{len(obj)}]"
        )

        if len(obj):

            recursive_print(
                obj[0],
                depth + 1,
                max_depth
            )

with open(
    "../../data/processed/CTB/prosit_params/sim_log_s5_sample_30.000_depth10_params.json",
    "r",
    encoding="utf-8"
) as f:

    model = json.load(f)

recursive_print(
    model["resource_params"]
)

resources
  LIST[26]
max_concurrency
  Res.GateIn
  Res.GateOut
  LL
  T13
  T14
  T24
  T16
  T17
  T15
  T12
  T21
  T20
  T18
  T11
  T23
  T27
  T22
  T10
  T06
  T09
  T07
  T08
  T26
  T19
  T25
  HO2
act_to_resources
  HO2_receive
    LIST[1]
  RMG_receive
    LIST[22]
  HO2_delivery
    LIST[1]
  Gate Out
    LIST[1]
  LL_receive
    LIST[1]
  RMG_mixed
    LIST[22]
  RMG_delivery
    LIST[22]
  HO2_mixed
    LIST[1]
  Gate In
    LIST[1]
  LL_mixed
    LIST[1]
  LL_delivery
    LIST[1]
resource_weights
  Res.GateIn
  Res.GateOut
  LL
  T13
  T14
  T24
    0
      feature
      threshold
      children
        true
        false
    1
      feature
      threshold
      children
        true
        false
    2
      value
    3
      value
    4
      feature
      threshold
      children
        true
        false
    5
      value
    6
      value
  T16
  T17
    0
      feature
      threshold
      children
        true
        false
    1
      feature
      threshold
 

In [22]:
features = set()

for res, tree_def in (
    model["resource_params"]
    ["resource_weights"]
    .items()
):

    if isinstance(tree_def, dict):

        for node in tree_def.values():

            if (
                isinstance(node, dict)
                and "feature" in node
            ):

                features.add(
                    node["feature"]
                )

print(
    sorted(features)
)

['has_reefer', 'rmg_utilization', 'target_utilization', 'target_utilization_bin = medium', 'vc_utilization']


In [23]:
# TREEEEE DARSTELLENNNN
from pprint import pprint

pprint(

    model["resource_params"]
    ["resource_weights"]
    ["T24"]

)

{'0': {'children': {'false': 4, 'true': 1},
       'feature': 'target_utilization',
       'threshold': 0.576},
 '1': {'children': {'false': 3, 'true': 2},
       'feature': 'vc_utilization',
       'threshold': 0.35},
 '2': {'value': 0.033},
 '3': {'value': 0.0},
 '4': {'children': {'false': 6, 'true': 5},
       'feature': 'target_utilization',
       'threshold': 0.76},
 '5': {'value': 0.068},
 '6': {'value': 0.197}}


In [25]:
import joblib

obj = joblib.load(
    "../../data/processed/CTB/prosit_params/s5_no_target_block_params.pkl"
)

print(type(obj))

<class 'prosit.simulator.SimulatorParameters'>
